# C3 — Barnes-Hut en CUDA (Colab, T4)

Antes de correr: **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**. Correr las celdas en orden. Al terminar: **Archivo → Descargar → .ipynb** y guardarlo en `cuda/bh_cuda.ipynb` del repo (el enunciado pide el notebook ya ejecutado, con salidas).

## 1. La GPU asignada

In [ ]:
!nvidia-smi

## 2. Código
Se clona la rama `impl/cuda` (el repo es público). `make` compila `c_core/` con gcc y `cuda/bh_cuda.cu` con nvcc.

In [ ]:
!rm -rf barnes-hut-parallel && git clone -q -b impl/cuda https://github.com/samurestrepo/barnes-hut-parallel.git
%cd barnes-hut-parallel
!git log --oneline -3
!make -C cuda

## 3. Verificador (umbrales de float32)
Las 5 pruebas del contrato, con 128 hilos por bloque.

In [ ]:
!python common/verify.py correr --cmd "./cuda/bh_cuda" --float32 --workers 128

## 4. compute-sanitizer
`memcheck` busca accesos fuera de los arreglos y memoria sin inicializar. `racecheck` solo revisa memoria **compartida** (`__shared__`), que este diseño no usa: la pila de cada hilo es local, por eso se espera 0 avisos.

In [ ]:
!compute-sanitizer --tool memcheck ./cuda/bh_cuda --input data/uniform_N2000_s42.txt --steps 2 --workers 128 --out /tmp/cs 2>&1 | tail -5
!compute-sanitizer --tool racecheck ./cuda/bh_cuda --input data/uniform_N2000_s42.txt --steps 1 --workers 128 --out /tmp/cs 2>&1 | tail -3

## 5. Desglose por fase y divergencia de warps
N = 200 000, 20 pasos. `ORDEN_HILOS=morton` (vecinos en el espacio en el mismo warp) contra `original` (warps mezclados).

In [ ]:
!python common/gen.py --dist uniform --n 200000
!python common/gen.py --dist corner --n 200000
!mkdir -p results/cuda
for dist, dt in [("uniform", "0.001"), ("corner", "2.5e-4")]:
    for orden in ["morton", "original"]:
        pref = f"results/cuda/{dist}_N200000_{orden}_b128"
        !ORDEN_HILOS={orden} ./cuda/bh_cuda --input data/{dist}_N200000_s42.txt --steps 20 --dt {dt} --workers 128 --out {pref}
        !cat {pref}_times.csv
!rm -f results/cuda/*_acc0.txt results/cuda/*_final.txt

## 6. Tamaño de bloque
64, 128 y 256 hilos por bloque con `common/bench.py` (de Samuel).

In [ ]:
!python common/bench.py --impl cuda --etiqueta uniform_N200000_bloques --cmd "./cuda/bh_cuda" --input data/uniform_N200000_s42.txt --trabajadores 64,128,256
!cat results/cuda/uniform_N200000_bloques_bench.csv

## 7. Comprimir resultados para subirlos al repo
Descargar `resultados_cuda.zip` y descomprimirlo en la raíz del repo.

In [ ]:
!cp results/lscpu.txt results/cuda/lscpu_colab.txt; zip -qr resultados_cuda.zip results/cuda && ls -la resultados_cuda.zip
from google.colab import files
files.download("resultados_cuda.zip")